In [2]:
import folium

import os, json, grpc, math, collections, hashlib, folium, pandas as pd
from sqlalchemy import create_engine
from senzing import SzEngineFlags as F
from senzing_grpc import SzAbstractFactoryGrpc

HYATT = 2120

In [3]:
channel = grpc.insecure_channel(
    f"{os.getenv('SENZING_GRPC_HOST','resolver')}:8261",
    options=[('grpc.max_receive_message_length', 200*1024*1024)])
grpc.channel_ready_future(channel).result(timeout=20)
engine = SzAbstractFactoryGrpc(channel).create_engine()
print("✅ connected")

✅ connected


In [4]:
db = create_engine(
    f"postgresql+psycopg2://{os.getenv('POSTGRES_USER','postgres')}:"
    f"{os.getenv('POSTGRES_PASSWORD','workshop')}@"
    f"{os.getenv('POSTGRES_HOST','postgres')}:5432/{os.getenv('POSTGRES_DB','erkg')}")
print("✅ connected")

✅ connected


In [5]:
geo = pd.read_sql("""
    SELECT entity_id, street, city, placekey, latitude, longitude
    FROM workshop.entity_geo WHERE street NOT LIKE 'PO BOX%%'
""", db)

pos, addr, entity_place = {}, collections.defaultdict(list), {}
for r in geo.itertuples():
    eid = int(r.entity_id)
    if eid not in pos:                       # first row wins, and we remember
        pos[eid] = (r.latitude, r.longitude) #   WHICH placekey it came from
        entity_place[eid] = r.placekey
    addr[eid].append((r.street, r.city, r.placekey))

print(f"{len(pos):,} entities have coordinates")

2,376 entities have coordinates


In [6]:
info, edges = {}, []

handle = engine.export_json_entity_report(F.SZ_EXPORT_DEFAULT_FLAGS)
try:
    while (line := engine.fetch_next(handle)):
        d = json.loads(line)
        e = d['RESOLVED_ENTITY']
        a = e['ENTITY_ID']
        if a not in pos:
            continue

        recs = e.get('RECORDS', [])
        info[a] = {
            'name':       e.get('ENTITY_NAME', ''),
            'n_records':  len(recs),
            'sources':    sorted({r['DATA_SOURCE'] for r in recs}),
            'record_ids': [f"{r['DATA_SOURCE']}:{r['RECORD_ID']}" for r in recs[:6]],
            'n_related':  len(d.get('RELATED_ENTITIES', [])),
        }

        for r in d.get('RELATED_ENTITIES', []):
            b = r['ENTITY_ID']
            if b in pos and a < b and pos[a] != pos[b]:
                edges.append({
                    'a': a, 'b': b,
                    'match_key':  r['MATCH_KEY'],
                    'level':      r.get('MATCH_LEVEL_CODE', ''),
                    'errule':     r.get('ERRULE_CODE', ''),
                    'ambiguous':  bool(r.get('IS_AMBIGUOUS', 0)),
                    'disclosed':  bool(r.get('IS_DISCLOSED', 0)),
                })
finally:
    engine.close_export_report(handle)

import collections
print(f"{len(info):,} entities, {len(edges)} relationships spanning locations")
print(collections.Counter(e['match_key'] for e in edges))

2,376 entities, 45 relationships spanning locations
Counter({'+NAME': 35, '+ADDRESS': 9, '+NAME+ADDRESS (Ambiguous)': 1})


In [7]:
# validated: orange and violet separate under CVD and clear 3:1 on a light
# basemap. The ambiguous/+PHONE line takes a neutral so crimson is free to mean
# "liquor licence" on nodes - it is one edge out of 45, and the dash pattern
# distinguishes it anyway.
EDGE_STYLE = {
    '+NAME':    ('#eb6834', None),        # orange, solid
    '+ADDRESS': ('#4a3aa7', '8,6'),       # violet, dashed
    '+PHONE':   ('#52514e', '2,6'),       # neutral, dotted
}
AMBIGUOUS = ('#52514e', '2,6')
FALLBACK  = ('#52514e', '4,4')

def edge_style(e):
    if e['ambiguous']:
        return AMBIGUOUS
    return EDGE_STYLE.get(e['match_key'], FALLBACK)

def metres(a, b):
    (la1, lo1), (la2, lo2) = pos[a], pos[b]
    return 6371000 * 2 * math.asin(math.sqrt(
        math.sin(math.radians(la2-la1)/2)**2 +
        math.cos(math.radians(la1)) * math.cos(math.radians(la2)) *
        math.sin(math.radians(lo2-lo1)/2)**2))

def edge_tooltip(e):
    rows = [
        f"<b style='font-size:13px'>{e['match_key']}</b>",
        f"<i>{e['level']}</i> &middot; rule {e['errule']}"
        + (" &middot; <b>ambiguous</b>" if e['ambiguous'] else "")
        + (" &middot; disclosed" if e['disclosed'] else ""),
        "<hr style='margin:3px 0'>",
        f"<b>{info[e['a']]['name'][:42]}</b><br>&nbsp;{addr[e['a']][0][0]}, {addr[e['a']][0][1]}",
        "&darr;",
        f"<b>{info[e['b']]['name'][:42]}</b><br>&nbsp;{addr[e['b']][0][0]}, {addr[e['b']][0][1]}",
        f"<hr style='margin:3px 0'><b>{metres(e['a'], e['b']):,.0f} m</b> apart",
    ]
    return folium.Tooltip("<div style='max-width:340px'>" + "<br>".join(rows) + "</div>",
                          sticky=True)

def tooltip(eid):
    i, a = info[eid], addr[eid]
    rows = [
        f"<b style='font-size:13px'>{i['name'][:46]}</b>",
        f"entity {eid} &middot; {i['n_records']} record(s) &middot; {i['n_related']} related",
        "<hr style='margin:3px 0'>",
        "<b>sources:</b> " + ", ".join(i['sources']),
        "<b>address:</b><br>" + "<br>".join(f"&nbsp;{s}, {c}" for s, c, _ in a[:3]),
        f"<b>placekey:</b> {a[0][2]}",
        "<b>records:</b><br>" + "<br>".join(f"&nbsp;{r}" for r in i['record_ids']),
    ]
    return folium.Tooltip("<div style='max-width:320px'>" + "<br>".join(rows) + "</div>",
                          sticky=True)

print(tooltip(HYATT).text[:200])


<div style='max-width:320px'><b style='font-size:13px'>HYATT REGENCY SAN FRANCISCO AIRPORT</b><br>entity 2120 &middot; 3 record(s) &middot; 5 related<br><hr style='margin:3px 0'><br><b>sources:</b> AB


In [8]:
# one marker per LOCATION, not per entity - entities at the same address share
# a coordinate exactly, so they were previously stacked invisibly on top of
# each other. Size encodes how many are there.
at_location = collections.defaultdict(list)
for eid, latlon in pos.items():
    at_location[latlon].append(eid)

on_relationship = {e['a'] for e in edges} | {e['b'] for e in edges}

def radius(n):
    return min(3 * math.sqrt(n), 22)      # sqrt so AREA tracks the count

def location_tooltip(latlon):
    eids = at_location[latlon]
    streets = sorted({addr[e][0][0] for e in eids})
    rows = [f"<b style='font-size:13px'>{len(eids)} business(es) here</b>",
            "<b>address:</b> " + "; ".join(streets[:3]),
            "<hr style='margin:3px 0'>"]
    rows += [f"&nbsp;{info.get(e, {}).get('name', '?')[:40]}" for e in eids[:10]]
    if len(eids) > 10:
        rows.append(f"&nbsp;<i>+{len(eids)-10} more</i>")
    return folium.Tooltip("<div style='max-width:320px'>" + "<br>".join(rows) + "</div>",
                          sticky=True)

sizes = sorted((len(v) for v in at_location.values()), reverse=True)
print(f"{len(at_location)} locations; busiest has {sizes[0]} businesses, median {sizes[len(sizes)//2]}")

452 locations; busiest has 117 businesses, median 2


In [9]:
import hashlib

# aggregate by PLACEKEY. Many placekeys share a hex centroid, so they would stack
# invisibly - spread them deterministically inside their own hex. The position is
# only hex-accurate anyway, so this loses no precision that was ever there.
at_place, place_pos, place_addr = collections.defaultdict(list), {}, {}
for r in geo.itertuples():
    eid = int(r.entity_id)
    # an entity can have several address spellings that geocode to the SAME
    # placekey (TOPGOLF: "258 N ANZA BLVD" and "258 ANZA BLVD") - count it once
    if eid not in at_place[r.placekey]:
        at_place[r.placekey].append(eid)
    place_pos.setdefault(r.placekey, (r.latitude, r.longitude))
    place_addr.setdefault(r.placekey, (r.street, r.city))

JITTER_M = 35

def jitter(pk, lat, lon):
    h = int(hashlib.md5(pk.encode()).hexdigest()[:8], 16)      # deterministic
    ang = (h % 3600) / 3600 * 2 * math.pi
    rad = ((h >> 12) % 1000) / 1000 * JITTER_M
    return (lat + (rad * math.cos(ang)) / 111320,
            lon + (rad * math.sin(ang)) / (111320 * math.cos(math.radians(lat))))

place_xy = {pk: jitter(pk, *place_pos[pk]) for pk in at_place}

on_relationship = {e['a'] for e in edges} | {e['b'] for e in edges}

# short labels so a tooltip listing ten businesses stays readable
SOURCE_SHORT = {
    'BURLINGAME':     'BUS',
    'ABC-RETAIL':     'ABC',
    'ABC-NONRETAIL':  'ABC-NR',
    'CA-CONTRACTORS': 'CON',
}
ABC_SOURCES = {'ABC-RETAIL', 'ABC-NONRETAIL'}

def badge(src):
    colour = '#c2185b' if src in ABC_SOURCES else '#52514e'
    return (f"<span style='background:{colour};color:#fff;padding:0 4px;"
            f"border-radius:3px;font-size:10px'>{SOURCE_SHORT.get(src, src)}</span>")

def place_tooltip(pk):
    eids = at_place[pk]
    street, city = place_addr[pk]
    datasets = sorted(set().union(*(set(info[e]['sources']) for e in eids)))
    rows = [
        f"<b style='font-size:13px'>{len(eids)} business(es) here</b>",
        f"{street}, {city}",
        "<b>datasets:</b> " + " ".join(badge(s) for s in datasets),
        f"<span style='color:#52514e;font-size:11px'>{pk}</span>",
        "<hr style='margin:3px 0'>",
    ]
    for e in eids[:10]:
        n = info[e]['n_records']
        rows.append(f"&nbsp;{info[e]['name'][:30]} "
                    + " ".join(badge(s) for s in info[e]['sources'])
                    + f" <span style='color:#52514e;font-size:10px'>{n} rec</span>")
    if len(eids) > 10:
        rows.append(f"&nbsp;<i>+{len(eids)-10} more</i>")
    return folium.Tooltip("<div style='max-width:380px'>" + "<br>".join(rows) + "</div>",
                          sticky=True)

sizes = sorted((len(v) for v in at_place.values()), reverse=True)
print(f"{len(at_place):,} placekeys; busiest has {sizes[0]} businesses, "
      f"{sum(1 for s in sizes if s > 1)} have more than one")


1,855 placekeys; busiest has 87 businesses, 248 have more than one


In [10]:
from folium import DivIcon

place_sources = {pk: set().union(*(set(info[e]['sources']) for e in eids))
                 for pk, eids in at_place.items()}

ABC = {'ABC-RETAIL', 'ABC-NONRETAIL'}

def shape_for(sources):
    """Multiple datasets always wins the shape."""
    if len(sources) > 1:              return 'circle'
    if sources & ABC:                 return 'triangle'
    if 'CA-CONTRACTORS' in sources:   return 'diamond'
    return 'square'

# ABC is carried by COLOUR so it survives the multi-dataset rule - the Hyatt
# stays a circle (6 businesses, 2 datasets) but turns crimson.
ABC_RED     = '#c2185b'     # validated: >= 3.6:1 over every OSM tone, and
                            # separates from both edge hues under CVD
INK_LINKED  = '#0b0b0b'
INK_PLAIN   = '#8a8982'

def fill_for(sources, linked):
    if sources & ABC:
        return ABC_RED
    return INK_LINKED if linked else INK_PLAIN

def marker_size(n):
    return int(min(8 + 5.5 * math.sqrt(n), 48))

def svg_marker(shape, size, fill, opacity):
    s, half, stroke, sw = size, size / 2, '#ffffff', 1.5
    common = f'fill="{fill}" fill-opacity="{opacity}" stroke="{stroke}" stroke-width="{sw}"'
    if shape == 'triangle':
        body = f'<polygon points="{half},1 {s-1},{s-1} 1,{s-1}" {common}/>'
    elif shape == 'square':
        body = f'<rect x="1" y="1" width="{s-2}" height="{s-2}" {common}/>'
    elif shape == 'diamond':
        body = f'<polygon points="{half},1 {s-1},{half} {half},{s-1} 1,{half}" {common}/>'
    else:
        body = f'<circle cx="{half}" cy="{half}" r="{half-1.5}" {common}/>'
    return DivIcon(icon_size=(s, s), icon_anchor=(half, half),
                   html=f'<svg width="{s}" height="{s}" style="overflow:visible">{body}</svg>')

import collections as _c
print(_c.Counter(shape_for(s) for s in place_sources.values()))
print(f"{sum(1 for s in place_sources.values() if s & ABC)} placekeys hold a liquor licence (crimson)")
for n in (1, 5, 20, 91):
    print(f"  {n:>3} businesses -> {marker_size(n)}px")


Counter({'square': 1381, 'circle': 198, 'diamond': 183, 'triangle': 93})
190 placekeys hold a liquor licence (crimson)
    1 businesses -> 13px
    5 businesses -> 20px
   20 businesses -> 32px
   91 businesses -> 48px


### The first time you run this cell it can take several seconds.

In [11]:
fig = folium.Figure(width=1400, height=900)
m = folium.Map(location=list(pos[HYATT]), zoom_start=13, tiles='OpenStreetMap')
m.add_to(fig)

def entity_xy(eid):
    return place_xy[entity_place[eid]]

layers = {
    'triangle': folium.FeatureGroup(name='▲ Liquor licence only',     show=True),
    'circle':   folium.FeatureGroup(name='● Several datasets',        show=True),
    'diamond':  folium.FeatureGroup(name='◆ Contractor licence only', show=True),
    'square':   folium.FeatureGroup(name='■ Business licence only',   show=True),
}
rels = folium.FeatureGroup(name='Senzing relationships', show=True)

for pk, eids in at_place.items():
    sources = place_sources[pk]
    shape   = shape_for(sources)
    linked  = any(e in on_relationship for e in eids)
    folium.Marker(
        list(place_xy[pk]),
        icon=svg_marker(shape, marker_size(len(eids)),
                        fill_for(sources, linked),
                        0.9 if (linked or sources & ABC) else 0.55),
        tooltip=place_tooltip(pk)).add_to(layers[shape])

for e in edges:
    color, dash = edge_style(e)
    folium.PolyLine([list(entity_xy(e['a'])), list(entity_xy(e['b']))],
                    color=color, dash_array=dash,
                    weight=3, opacity=0.9, tooltip=edge_tooltip(e)).add_to(rels)

for g in layers.values():
    g.add_to(m)
rels.add_to(m)

folium.Marker(list(entity_xy(HYATT)),
              icon=folium.Icon(color='black', icon='star'),
              tooltip=tooltip(HYATT)).add_to(m)

folium.LayerControl(collapsed=False).add_to(m)

counts = collections.Counter(e['match_key'] for e in edges)
legend = f"""
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 9999;
     background: #fcfcfb; padding: 10px 14px; border: 1px solid #b5b4ad;
     border-radius: 4px; font: 12px/1.7 sans-serif; color: #0b0b0b;
     box-shadow: 0 1px 4px rgba(0,0,0,.3)">
  <b>Shape = which dataset</b><br>
  <svg width="16" height="14"><rect x="2" y="1" width="12" height="12" fill="#8a8982" stroke="#fff" stroke-width="1.5"/></svg>
    &nbsp;business licence only<br>
  <svg width="16" height="14"><polygon points="8,1 15,7 8,13 1,7" fill="#8a8982" stroke="#fff" stroke-width="1.5"/></svg>
    &nbsp;contractor licence only<br>
  <svg width="16" height="14"><polygon points="8,1 15,13 1,13" fill="#c2185b" stroke="#fff" stroke-width="1.5"/></svg>
    &nbsp;liquor licence only<br>
  <svg width="16" height="14"><circle cx="8" cy="7" r="6" fill="#8a8982" stroke="#fff" stroke-width="1.5"/></svg>
    &nbsp;several datasets<br>
  <hr style="margin:5px 0; border:0; border-top:1px solid #d8d7d0">
  <b><span style="color:#c2185b">Crimson</span> = a liquor licence is here</b>
  <span style="font-size:11px; color:#52514e">(any shape)</span><br>
  <b>Size</b> = businesses at that placekey<br>
  <svg width="90" height="30">
    <rect x="2"  y="16" width="11" height="11" fill="#8a8982" stroke="#fff" stroke-width="1.5"/>
    <rect x="22" y="9"  width="20" height="20" fill="#8a8982" stroke="#fff" stroke-width="1.5"/>
    <rect x="52" y="1"  width="32" height="32" fill="#8a8982" stroke="#fff" stroke-width="1.5"/>
  </svg><br>&nbsp;1 &nbsp;&nbsp;&nbsp;&nbsp; 5 &nbsp;&nbsp;&nbsp;&nbsp;&nbsp; 20+<br>
  <hr style="margin:5px 0; border:0; border-top:1px solid #d8d7d0">
  <b>Senzing relationship</b><br>
  <svg width="30" height="8"><line x1="0" y1="4" x2="30" y2="4" stroke="#eb6834" stroke-width="3"/></svg>
    &nbsp;+NAME ({counts.get('+NAME',0)})<br>
  <svg width="30" height="8"><line x1="0" y1="4" x2="30" y2="4" stroke="#4a3aa7" stroke-width="3" stroke-dasharray="8,6"/></svg>
    &nbsp;+ADDRESS ({counts.get('+ADDRESS',0)})<br>
  <svg width="30" height="8"><line x1="0" y1="4" x2="30" y2="4" stroke="#52514e" stroke-width="3" stroke-dasharray="2,6"/></svg>
    &nbsp;ambiguous ({sum(1 for e in edges if e['ambiguous'])})<br>
  <span style="color:#0b0b0b">&#9733;</span> Hyatt Regency SFO
</div>"""
m.get_root().html.add_child(folium.Element(legend))

fig


In [12]:
import json
from branca.element import MacroElement
from jinja2 import Template

class SearchBox(MacroElement):
    """A search box that flies the map to an entity by name or ID.
    Built as a MacroElement so its script renders AFTER the map is created."""

    _template = Template("""
{% macro html(this, kwargs) %}
<div id="erkg-search" style="position:absolute; top:12px; left:60px; z-index:9999;
     background:#fcfcfb; padding:8px 10px; border:1px solid #b5b4ad; border-radius:4px;
     font:13px sans-serif; box-shadow:0 1px 4px rgba(0,0,0,.3); width:300px">
  <input id="erkg-q" type="text" placeholder="entity name or ID, then Enter"
         style="width:100%; box-sizing:border-box; padding:5px; font:13px sans-serif">
  <div id="erkg-results" style="max-height:220px; overflow-y:auto; margin-top:4px"></div>
</div>
{% endmacro %}

{% macro script(this, kwargs) %}
(function() {
  var map  = {{ this._parent.get_name() }};
  var DATA = {{ this.data }};

  function go(e) {
    map.flyTo([e.lat, e.lon], 18);
    L.popup().setLatLng([e.lat, e.lon])
      .setContent('<b>' + e.name + '</b><br>entity ' + e.id).openOn(map);
  }

  function find(q) {
    q = q.trim().toUpperCase();
    if (!q) return [];
    if (/^\\d+$/.test(q)) return DATA.filter(function(e) { return String(e.id) === q; });
    return DATA.filter(function(e) {
      return e.name.toUpperCase().indexOf(q) !== -1; }).slice(0, 15);
  }

  var box   = document.getElementById('erkg-search'),
      input = document.getElementById('erkg-q'),
      out   = document.getElementById('erkg-results');
  L.DomEvent.disableClickPropagation(box);
  L.DomEvent.disableScrollPropagation(box);

  input.addEventListener('input', function() {
    var hits = find(input.value);
    out.innerHTML = '';
    hits.forEach(function(e) {
      var row = document.createElement('div');
      row.textContent = e.name + '  (' + e.id + ')';
      row.style.cssText = 'padding:3px 4px; cursor:pointer; border-bottom:1px solid #eee';
      row.onmouseover = function() { row.style.background = '#eee'; };
      row.onmouseout  = function() { row.style.background = ''; };
      row.onclick = function() { go(e); };
      out.appendChild(row);
    });
    if (input.value.trim() && !hits.length)
      out.innerHTML = '<i style="color:#888">no match</i>';
  });

  input.addEventListener('keydown', function(ev) {
    if (ev.key === 'Enter') { var h = find(input.value); if (h.length) go(h[0]); }
  });
})();
{% endmacro %}
""")

    def __init__(self, index):
        super().__init__()
        self._name = 'SearchBox'
        self.data = json.dumps(index)


In [13]:
# every entity that has a marker on the map, at the position its marker is
# actually drawn - jitter included, so the map flies to the right spot
search_index = [
    {'id':   eid,
     'name': info[eid]['name'],
     'lat':  place_xy[entity_place[eid]][0],
     'lon':  place_xy[entity_place[eid]][1]}
    for eid in info
    if eid in entity_place
]

print(f"{len(search_index):,} entities searchable")


2,376 entities searchable


In [14]:
SearchBox(search_index).add_to(m)
fig


In [15]:
FALLBACK_HEX = '5vg-7y7-f9f'

park = pd.read_sql("""
    SELECT entity_id, street, placekey FROM workshop.entity_geo
    WHERE placekey LIKE %(hex)s ORDER BY entity_id
""", db, params={'hex': f'%@{FALLBACK_HEX}'})

all_geo = pd.read_sql("SELECT entity_id, street, placekey FROM workshop.entity_geo", db)

rows = []
for eid, grp in park.groupby('entity_id'):
    e = json.loads(engine.get_entity_by_entity_id(int(eid)))['RESOLVED_ENTITY']
    real = all_geo[(all_geo.entity_id == eid) &
                   (~all_geo.placekey.str.endswith(f'@{FALLBACK_HEX}'))]
    rows.append({
        'entity_id':               int(eid),
        'name':                    e['ENTITY_NAME'],
        'records':                 len(e['RECORDS']),
        'sources':                 ', '.join(sorted({r['DATA_SOURCE'] for r in e['RECORDS']})),
        'park address':            '; '.join(grp.street),
        'also has a real address': '; '.join(real.street) if len(real) else '-',
    })

park_table = pd.DataFrame(rows)
park_table


,entity_id,name,records,sources,park address,also has a real address
0,35,836-868 COWAN RD LLC,1,BURLINGAME,836 COWAN RD,-
1,653,BENIHANA OF TOKYO INC,2,"ABC-RETAIL, BURLINGAME",1494 BAYSHORE HWY,1496 BAYSHORE HWY
2,1359,DEVINCENZI ARCHITECTURAL PRODUCTS GROUP INC,2,"BURLINGAME, CA-CONTRACTORS",1717 ADVIAN ROAD,1717 ADRIAN RD
3,1453,DYNASTY PROPERTIES COMPANY LTD,1,BURLINGAME,106- BROADWAY,-
4,1843,GIC BURLINGAME LLC,1,BURLINGAME,1775 ROLLINS RD,-
5,2076,HILLS POOL SERVICE INC,2,"BURLINGAME, CA-CONTRACTORS",23 EDWRADS CT,23 EDWARDS CT
6,2146,IMAGINE CULINARY GROUP INC,1,BURLINGAME,301 AIRPORT BLVD,-
7,2155,INCREDIFLIX INC,1,BURLINGAME,850 BULRINGAME AVE,-
8,2364,K AND M CUSTOM HOMES INC,2,"BURLINGAME, CA-CONTRACTORS",1425 HOWARD AVE #526,1325 HOWARD AVE #526
9,2367,K K ROSETTA CO,1,BURLINGAME,1710 ADRIAN RD # 10,-
